# Búsqueda de documentos con MinHash

----------------


**Objetivo General:**

Implementar y evaluar el algoritmo MinHash para la búsqueda eficiente de documentos similares en un conjunto de datos textuales, aprovechando técnicas de hashing sensible a la localidad (LSH) para reducir la complejidad computacional.



En esta libreta veremos cómo hacer búsqueda eficiente de documentos considerando la similitud de Jaccard.

La similitud de Jaccard entre un par de conjuntos $(\mathcal{C}^{(1)}, \mathcal{C}^{(2)})$ está dada por

$$
J(\mathcal{C}^{(1)}, \mathcal{C}^{(2)}) = \frac{\mid \mathcal{C}^{(1)} \cap \mathcal{C}^{(2)} \mid}{\mid \mathcal{C}^{(1)}\cup \mathcal{C}^{(2)} \mid} \in [0,1]
$$

In [ ]:
from collections import Counter
from math import floor, log
import codecs
import re

import numpy as np
from scipy.sparse import csr_matrix, lil_matrix
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

VOCMAX = 5000
n_muestras_sint = 10000
n_tablas_ng = 50


# Para reproducibilidad
np.random.seed(2021)

## Conjuntos de datos
Preparamos dos conjuntos de datos para probar los algoritmos de búsqueda.

### Datos sintéticos
Este conjunto de datos está compuesto por 5 listas de elementos y un conjunto universo de 10 elementos.


In [ ]:
sint_conj = [[0, 1, 4, 6, 8], [2, 3, 4, 7, 8], [1, 4, 6, 7], [0, 5, 6, 8], [0, 1, 3, 4, 7]]
univ = {e for l in sint_conj for e in l}

Calculamos la similitud de Jaccard de todos los pares.

In [ ]:
sims_jacc = np.identity(len(sint_conj))
for i in range(0, len(sint_conj) - 1):
  ci = set(sint_conj[i])
  for j in range(i + 1, len(sint_conj)):
    cj = set(sint_conj[j])
    sims_jacc[i,j] = float(len(ci.intersection(cj)) / len(ci.union(cj)))
    sims_jacc[j,i] = sims_jacc[i,j]

print(sims_jacc)

[[1.         0.25       0.5        0.5        0.42857143]
 [0.25       1.         0.28571429 0.125      0.42857143]
 [0.5        0.28571429 1.         0.14285714 0.5       ]
 [0.5        0.125      0.14285714 1.         0.125     ]
 [0.42857143 0.42857143 0.5        0.125      1.        ]]


Definimos una función que calcula la propiedad de colisión de todos los pares en este conjunto de datos

In [ ]:
def p_colision(db, tablas, n_tablas, n_ej):
    """
    Calcula la matriz de probabilidades de colisión entre documentos usando tablas LSH (MinHash).

    Esta función inserta todos los documentos de la base de datos en múltiples tablas hash LSH
    y luego estima la probabilidad de colisión entre cada par de documentos como la fracción de
    tablas en las que ambos documentos caen en la misma cubeta.

    Parámetros:
    -----------
    db : list
        Lista de documentos, donde cada documento está representado por su firma MinHash.
    tablas : list
        Lista de tablas hash LSH (objetos MinHashTable) previamente inicializadas.
    n_tablas : int
        Número de tablas hash a utilizar (debe ser igual a len(tablas)).
    n_ej : int
        Número de documentos en la base de datos (debe ser igual a len(db)).

    Retorna:
    --------
    numpy.ndarray
        Matriz simétrica de forma (n_ej, n_ej) donde la entrada [i, j] representa la
        probabilidad estimada de que los documentos i y j colisionen en una tabla LSH.
        Los valores están en el rango [0, 1].
    """

    # Insertar todos los documentos en cada tabla hash
    for i in range(n_tablas):
        for j, l in enumerate(db):
            tablas[i].insertar(l, j)

    # Inicializar matriz de conteo de colisiones
    colisiones = np.zeros((n_ej, n_ej))

    # Contar colisiones para cada par de documentos
    for i in range(n_tablas):
        for j, cj in enumerate(db):
            # Buscar documentos que colisionan con cj en la tabla i
            for e in tablas[i].buscar(cj):
                colisiones[j, e] += 1

    # Calcular probabilidades dividiendo por el número de tablas
    return colisiones / n_tablas

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
import nltk

recursos = ['punkt_tab', 'averaged_perceptron_tagger', 'wordnet', 'averaged_perceptron_tagger_eng']
for recurso in recursos:
    try:
        nltk.data.find(recurso)
    except LookupError:
        nltk.download(recurso)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


### 20 Newsgroups
Vamos a usar el conjunto de documentos de _20 Newsgropus_, el cual descargamos usando scikit-learn.

In [ ]:
db = fetch_20newsgroups(remove=('headers','footers','quotes'))

Importamos la biblioteca NLTK y definimos nuestro analizador léxico y lematizador

In [ ]:
import nltk
nltk.download(['punkt','averaged_perceptron_tagger','wordnet'])

from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet
from nltk.corpus.reader.wordnet import NOUN, VERB, ADV, ADJ

morphy_tag = {
    'JJ' : ADJ,
    'JJR' : ADJ,
    'JJS' : ADJ,
    'VB' : VERB,
    'VBD' : VERB,
    'VBG' : VERB,
    'VBN' : VERB,
    'VBP' : VERB,
    'VBZ' : VERB,
    'RB' : ADV,
    'RBR' : ADV,
    'RBS' : ADV
}

def doc_a_tokens(doc):
  tagged = pos_tag(word_tokenize(doc.lower()))
  lemmatizer = WordNetLemmatizer()
  tokens = []
  for p,t in tagged:
    tokens.append(lemmatizer.lemmatize(p, pos=morphy_tag.get(t, NOUN)))

  return tokens

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Convertimos el conjunto preprocesado a una lista de cadenas, una por documento

In [ ]:
corpus = []
for d in db.data:
  d = d.replace('\n',' ').replace('\r',' ').replace('\t',' ')
  d = ' '.join([''.join([c.lower() for c in p if c.isalnum()]) for p in d.split()])
  tokens = doc_a_tokens(d)
  corpus.append(' '.join(tokens))

Dividimos nuestro conjunto en 2 subconjuntos: los documentos de la base que se buscarán y los documentos de consulta

In [ ]:
perm = np.random.permutation(len(corpus)).astype(int)
n_ej_base = int(floor(len(corpus) * 0.95))

base = [corpus[i] for i in perm[:n_ej_base]]
consultas = [corpus[i] for i in perm[n_ej_base:]]

Obtenemos y cargamos la lista de _stopwords_ para inglés (archivo con una palabra por línea)

In [ ]:
!wget -qO- -O stopwords_english.txt \
         https://raw.githubusercontent.com/pan-webis-de/authorid/master/data/stopwords_english.txt

stopwords = []
for line in codecs.open('stopwords_english.txt', encoding = "utf-8"):
  stopwords.append(line.rstrip())

Procesamos cada palabra del corpus completo para generar y ordenar el vocabulario

In [ ]:
# Divide la cadena en palabras
term_re = re.compile("\w+", re.UNICODE)

# Contamos las ocurrencias de cada palabra
corpus_freq = Counter()
doc_freq = Counter()
for d in base:
  # Eliminamos números de la cadena (documento) a procesar
  d = re.sub(r'\d+', '', d)

  # Dividimos la cadena en una lista de palabras
  terms = [t for t in term_re.findall(d) if t not in stopwords and len(t) > 2]

  # Aumentamos el contador de cada instancia palabra en el documento
  for t in terms:
    corpus_freq[t] += 1

  # Aumentamos el contador de cada palabra distinta en el documento
  for t in set(terms):
    doc_freq[t] += 1

# Generamos un diccionario con las VOCMAX palabras más frecuentes
vocabulary = {entry[0]:(i, entry[1], doc_freq[entry[0]], log(len(corpus) / doc_freq[entry[0]])) \
              for i, entry in enumerate(corpus_freq.most_common()) \
              if i < VOCMAX}

Creamos un diccionario para mapear índices a palabras

In [ ]:
id_a_palabra = {v[0]: k for k,v in vocabulary.items()}

Generamos las bolsas de palabras de los documentos preprocesados

In [ ]:
def cadenas_a_bolsas(cadenas, voc, descartar, tre):
  bolsas = []
  for c in cadenas:
    c = re.sub(r'\d+', '', c)
    ids = Counter([voc[t][0] for t in tre.findall(c) \
                   if t in voc and t not in descartar])
    bolsas.append([i for i in sorted(ids.items())])

  return bolsas

bolsas_base = cadenas_a_bolsas(base, vocabulary, stopwords, term_re)
bolsas_consultas = cadenas_a_bolsas(consultas, vocabulary, stopwords, term_re)

In [ ]:
def csr_a_ldb(csr):
    """
    Convierte una matriz CSR (Compressed Sparse Row) a una lista de bolsas de palabras (LDB).

    Parámetros:
    -----------
    csr : scipy.sparse.csr_matrix
        Matriz dispersa en formato CSR, donde cada fila representa un documento y las columnas
        son características (ej: palabras en un vocabulario). Los valores distintos de cero
        indican la presencia o peso de la característica.

    Retorna:
    --------
    list
        Lista de listas (LDB) donde cada sublista contiene los índices de las columnas
        con valores distintos de cero para cada documento (fila).
        Formato: [[idx1, idx2, ...], [idx3, idx4, ...], ...]
    """
    ldb = [[] for _ in range(csr.shape[0])]  # Inicializa una lista vacía por documento
    coo = csr.tocoo()  # Convierte a formato COO para iterar eficientemente
    for i, j, v in zip(coo.row, coo.col, coo.data):
        ldb[i].append(j)  # Agrega el índice de la columna (j) al documento (i)
    return ldb

def ldb_a_csr(ldb, dim):
    """
    Convierte una lista de bolsas de palabras (LDB) a una matriz CSR.

    Parámetros:
    -----------
    ldb : list
        Lista de listas donde cada sublista contiene tuplas (índice_columna, valor) para un documento.
        Formato: [[(idx1, val1), (idx2, val2), ...], [(idx3, val3), ...], ...]
    dim : int
        Dimensión de la matriz de salida (número de columnas, ej: tamaño del vocabulario).

    Retorna:
    --------
    scipy.sparse.csr_matrix
        Matriz dispersa en formato CSR, con shape=(len(ldb), dim).
    """
    n_el = sum(len(l) for l in ldb)  # Calcula el total de elementos no cero

    # Inicializa arrays para COO
    vals = np.zeros(n_el, dtype=float)
    rows = np.zeros(n_el, dtype=int)
    cols = np.zeros(n_el, dtype=int)

    # Llena los arrays
    j = 0
    for i, l in enumerate(ldb):
        for e in l:
            cols[j] = e[0]  # Índice de columna
            vals[j] = e[1]  # Valor (ej: TF-IDF)
            rows[j] = i     # Índice de fila (documento)
            j += 1

    return csr_matrix((vals, (rows, cols)), shape=(len(ldb), dim))

bolsas_base_csr = ldb_a_csr(bolsas_base, VOCMAX)
bolsas_consultas_csr = ldb_a_csr(bolsas_consultas, VOCMAX)

In [ ]:
print(bolsas_base_csr[9, :])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 206)	1.0
  (0, 276)	1.0
  (0, 822)	1.0
  (0, 1998)	2.0
  (0, 2088)	1.0


## MinHash binario
Min-Hashing es un algoritmo para búsqueda de conjuntos similares bajo la similitud de Jaccard, la cual consiste en:
* Generar permutación aleatoria del conjunto universo $\mathbb{U}$
* Asignar a cada conjunto su 1er elemento bajo la permutación, esto es,
$$
h(\mathcal{C}^{(i)}) = min(\pi(\mathcal{C}^{(i)}))
$$

La probabilidad de que dos conjuntos tengan valor MinHash idéndico es igual a su similitud de Jaccard:
$$
P[h(\mathcal{C}^{(i)}) = h(\mathcal{C}^{(j)})] = \frac{\mid \mathcal{C}^{(i)} \cap \mathcal{C}^{(j)} \mid}{\mid \mathcal{C}^{(i)}\cup \mathcal{C}^{(j)} \mid} \in [0,1]
$$

Para buscar conjuntos similares, los valores MinHash se agrupan en $l$ tuplas de
$r$ funciones distintas de la siguiente forma:    

\begin{align*}
 g_1(\mathcal{C}^{(i)}) & = (h_1(\mathcal{C}^{(i)}), h_2(\mathcal{C}^{(i)}), \ldots , h_r(\mathcal{C}^{(i)}))\\
g_2(\mathcal{C}^{(i)}) & = (h_{r+1}(\mathcal{C}^{(i)}), h_{r+2}(\mathcal{C}^{(i)}), \ldots , h_{2\cdot r}(\mathcal{C}^{(i)}))\\
      \cdots\\
g_l(\mathcal{C}^{(i)}) & = (h_{(l-1)\cdot r+1}(\mathcal{C}^{(i)}), h_{(l-1)\cdot r2}(\mathcal{C}^{(i)}), \ldots , h_{l\cdot r}(\mathcal{C}^{(i)}))
\end{align*}
    
Conjuntos con una tupla idéntica se almacenan en la misma cubeta en la tabla asociada a la tupla.

In [ ]:
class MinHashTable:
    """
    Implementación de una tabla hash sensible a la localidad (LSH) basada en MinHash.

    Esta clase permite indexar y buscar eficientemente conjuntos de datos similares utilizando
    la técnica de MinHash con múltiples funciones hash. Los elementos similares se mapean a
    las mismas cubetas con alta probabilidad, permitiendo búsquedas aproximadas de vecinos
    cercanos.

    Atributos:
        n_cubetas (int): Número de cubetas en la tabla hash.
        tabla (list): Estructura de almacenamiento para las cubetas.
        dim (int): Dimensión del espacio de entrada (número de características únicas).
        t_tupla (int): Número de funciones hash MinHash utilizadas.
        perm (ndarray): Matriz de permutaciones aleatorias para simular funciones hash.
        rind (ndarray): Matriz de reindexación aleatoria para evitar colisiones.
        a, b (ndarray): Coeficientes aleatorios para combinar hashes.
        primo (int): Número primo grande para la función hash.
    """

    def __init__(self, n_cubetas, t_tupla, dim):
        """
        Inicializa una nueva tabla hash MinHash.

        Args:
            n_cubetas (int): Número de cubetas en la tabla hash.
            t_tupla (int): Número de funciones hash MinHash a utilizar.
            dim (int): Dimensión del espacio de entrada (número de características únicas).
        """
        self.n_cubetas = n_cubetas
        self.tabla = [[] for _ in range(n_cubetas)]
        self.dim = dim
        self.t_tupla = t_tupla

        # Matrices aleatorias para las funciones hash
        self.perm = np.random.uniform(0, 1, size=(self.t_tupla, self.dim))
        self.rind = np.random.randint(0, np.iinfo(np.int32).max, size=(self.t_tupla, self.dim))

        # Coeficientes aleatorios para combinar hashes
        self.a = np.random.randint(0, np.iinfo(np.int32).max, size=self.t_tupla)
        self.b = np.random.randint(0, np.iinfo(np.int32).max, size=self.t_tupla)
        self.primo = 4294967291  # Número primo grande

    def __repr__(self):
       """
       Representación formal de la tabla hash.

       Returns:
        str: Cadena que representa la tabla con todas las cubetas.
      """
       contenido = "\n".join(f"{i}::{repr(self.tabla[i])}" for i in range(self.n_cubetas))
       return f"<TablaHash:\n{contenido}>"




    def __str__(self):
        """
        Representación legible de la tabla hash (solo cubetas no vacías).

        Returns:
            str: Cadena que muestra las cubetas no vacías.
        """
        contenido = [f'{i}::{self.tabla[i]}' for i in range(self.n_cubetas) if self.tabla[i]]
        return '\n'.join(contenido)

    def h(self, x):
        """
        Función hash usando módulo primo.

        Args:
            x (int): Valor a hashear.

        Returns:
            int: Hash resultante.
        """
        return x % self.primo

    def sl(self, x, i):
        """
        Sondeo lineal para manejo de colisiones.

        Args:
            x (int): Valor original a hashear.
            i (int): Intento actual de sondeo.

        Returns:
            int: Posición en la tabla hash para el i-ésimo intento.
        """
        return (self.h(x) + i) % self.n_cubetas

    def minhash(self, x):
        """
        Calcula la firma MinHash para un conjunto de características.

        Args:
            x (array-like): Índices de características activas en el conjunto.

        Returns:
            tuple: Par de hashes (mh, v2) donde:
                - mh: Firma MinHash principal para agrupación.
                - v2: Hash secundario para manejo de colisiones.

        """

        # Extraemos los valores de permutaciones de acuerdo al indice de x (todas las filas, solo las columnas que correspondan al indice x)
        xp = self.perm[:, x]

        # Extraemos los valores unicos de acuerdo al indice de x (todas las filas, solo las columnas que correspondan al indice x)
        xi = self.rind[:, x]

        # Obtenemos el indice del elemento minimo de la permutación por cada fila
        amin = xp.argmin(axis=1)

        # Obtenemos un vector con todos los elementos unicos de todas las funciones hash
        emin = xi[np.arange(self.t_tupla), amin]

        return (np.sum(self.a * emin, dtype=np.ulonglong),
                np.sum(self.b * emin, dtype=np.ulonglong))

    def insertar(self, x, ident):
        """
        Inserta un conjunto en la tabla hash asociado a un identificador.

        Args:
            x (array-like): Índices de características del conjunto a insertar.
            ident: Identificador único del conjunto.

        Raises:
            Imprime mensaje de error si la tabla está llena.
        """

        mh, v2 = self.minhash(x)
        llena = True

        for i in range(self.n_cubetas):
            cubeta = int(self.sl(v2, i))
            if not self.tabla[cubeta]:
                self.tabla[cubeta].append(mh)
                self.tabla[cubeta].append([ident])
                llena = False
                break
            elif self.tabla[cubeta][0] == mh:
                self.tabla[cubeta][1].append(ident)
                llena = False
                break

        if llena:
            print('¡Error, tabla llena!')

    def buscar(self, x):
        """
        Busca conjuntos similares al conjunto de consulta.

        Args:
            x (array-like): Índices de características del conjunto consulta.

        Returns:
            list: Identificadores de conjuntos similares encontrados.
        """
        mh, v2 = self.minhash(x)

        for i in range(self.n_cubetas):
            cubeta = int(self.sl(v2, i))
            if not self.tabla[cubeta]:
                return []
            elif self.tabla[cubeta][0] == mh:
                return self.tabla[cubeta][1]

        return []

    def eliminar(self, x, ident):
        """
        Elimina un identificador asociado a un conjunto de la tabla.

        Args:
            x (array-like): Índices de características del conjunto.
            ident: Identificador a eliminar.

        Returns:
            int: -1 si no se encontró el elemento, None si se eliminó correctamente.
        """
        mh, v2 = self.minhash(x)

        for i in range(self.n_cubetas):
            cubeta = int(self.sl(v2, i))
            if not self.tabla[cubeta]:
                break
            elif self.tabla[cubeta][0] == mh:
                try:
                    self.tabla[cubeta][1].remove(ident)
                    return None
                except ValueError:
                    pass

        return -1


### Verificación con conjunto de datos sintéticos
Primero verificamos la probabilidad de colisión de dos conjuntos en nuestra implementación.

In [ ]:
tablas_sint_bin = [MinHashTable(2**4, 1, len(univ)) for _ in range(n_muestras_sint)]
print(p_colision(sint_conj, tablas_sint_bin, n_muestras_sint, len(sint_conj)))

[[1.     0.2471 0.5046 0.5002 0.4253]
 [0.2471 1.     0.2827 0.1235 0.4242]
 [0.5046 0.2827 1.     0.1477 0.5002]
 [0.5002 0.1235 0.1477 1.     0.1224]
 [0.4253 0.4242 0.5002 0.1224 1.    ]]


### Búsqueda de documentos similares con _20 newsgroups_
Probamos la implementación de Min-Hashing para la búsqueda de documentos similares en _20 newsgroups_. Para realizar la búsqueda de documentos similares:

1. Insertamos las listas a nuestras tablas
2. Recuperamos los documentos similares a nuestros documentos de consulta usando las tablas MinHash.
3. Calculamos la similitud Jaccard de los documentos recuperados con los de consulta
4. Ordenamos por similitud.

In [ ]:
def similitud_jaccard(x, y):
  x = x.toarray()[0]
  y = y.toarray()[0]
  inter = np.count_nonzero(x * y)
  return inter / (np.count_nonzero(x) + np.count_nonzero(y) - inter)


def similitud_minmax(x, y):
  x = x.toarray()[0]
  y = y.toarray()[0]
  min = np.min(np.array([x, y]).T, axis=1).sum()
  max = np.max(np.array([x, y]).T, axis=1).sum()
  return min / max

def similitud_minmax_pesado(x, y, w):
  x = x.toarray()[0]
  y = y.toarray()[0]
  min = (w * np.min(np.array([x, y]).T, axis=1)).sum()
  max = (w * np.max(np.array([x, y]).T, axis=1)).sum()
  return min / max

def fuerza_bruta(ds, qs, fs):
  medidas = np.zeros(ds.shape[0])
  for i,x in enumerate(ds):
    medidas[i] = fs(qs, x)
  return np.sort(medidas)[::-1], np.argsort(medidas)[::-1]

def busca_pares_documentos(base_csr, consultas_csr, base_ldb, consultas_ldb, tablas, fs):
    """
    Busca documentos similares entre una base de datos y un conjunto de consultas utilizando LSH (Locality-Sensitive Hashing)
    y refina los resultados con comparación por fuerza bruta.

    El proceso consta de tres etapas principales:
    1. Indexación de los documentos base en las tablas hash LSH
    2. Búsqueda aproximada de candidatos similares para cada consulta
    3. Refinamiento de resultados mediante comparación exacta por fuerza bruta

    Parámetros:
        base_csr (scipy.sparse.csr_matrix): Matriz CSR que representa los documentos base.
                                           Cada fila es un documento en formato vectorial (ej: TF-IDF).
        consultas_csr (scipy.sparse.csr_matrix): Matriz CSR que representa las consultas.
        base_ldb (list): Lista de representaciones compactas (firmas LSH) de los documentos base.
        consultas_ldb (list): Lista de representaciones compactas de las consultas.
        tablas (list): Lista de tablas hash LSH (objetos MinHashTable) previamente inicializadas.
        fs (function): Función de similitud para la comparación por fuerza bruta (ej: similitud coseno).

    Retorna:
        tuple: Dos listas:
            - sims: Lista de listas con valores de similitud para cada consulta con sus documentos candidatos.
            - orden: Lista de listas con los índices de los documentos candidatos ordenados por similitud.

    Proceso:
        1. Fase de indexación:
           - Inserta todas las firmas LSH de los documentos base en las tablas hash.
        2. Fase de búsqueda:
           - Para cada consulta, recupera documentos candidatos de las tablas hash.
           - Elimina duplicados usando conjuntos (set).
        3. Fase de refinamiento:
           - Para cada consulta y sus candidatos, calcula similitudes exactas usando fuerza bruta.
           - Ordena los resultados por similitud descendente.

    Notas:
        - Las tablas hash deben estar previamente inicializadas con los mismos parámetros.
        - La función fs debe aceptar dos matrices CSR y retornar similitudes e índices ordenados.
        - Si no se encuentran candidatos para una consulta, retorna listas vacías para esa consulta.
    """
    # Indexación de documentos base
    for j, l in enumerate(base_ldb):
        for i in range(len(tablas)):
            if l:
                tablas[i].insertar(l, j)

    # Búsqueda de candidatos (documentos similares) para cada consulta
    docs = []
    for j, l in enumerate(consultas_ldb):
        dc = []
        if l:
            for i in range(len(tablas)):
                dc.extend(tablas[i].buscar(l))
        docs.append(set(dc))  # Eliminamos duplicados

    # Refinamiento por fuerza bruta (calculamos similitud para todos los documentos similares de consulta)
    sims = []
    orden = []
    for i, q in enumerate(consultas_csr):
        ld = list(docs[i])
        if ld:
            s, o = fuerza_bruta(base_csr[ld], q, fs)
            sims.append(s)
            orden.append([ld[e] for e in o])
        else:
            sims.append([])
            orden.append([])

    return sims, orden

Buscamos documentos similares y examinamos un ejemplo de consulta y su correspondiente documento más similar.

In [ ]:
tablas_ng_bin = [MinHashTable(2**18, 2, VOCMAX) for _ in range(n_tablas_ng)]
sims, orden = busca_pares_documentos(bolsas_base_csr,
                                     bolsas_consultas_csr,
                                     bolsas_base,
                                     bolsas_consultas,
                                     tablas_ng_bin,
                                     similitud_jaccard)
print("------ C O N S U L T A ------\n", consultas[21])
print("\n------ M Á S  S I M I L A R ------\n", base[list(orden[21])[0]])

------ C O N S U L T A ------
 actually the book be call seventh day adventist believe and there be 27 basica belief i believe it be print by the reveiew and herald publishing association competition be the law of the jungle cooperation be the law of civilization eldridge cleaver

------ M Á S  S I M I L A R ------
 the book be call 27 basic fundamental belief or something very close to that the number be 27 not 30 i have a copy at home im away at school


## Ejercicio en clase
+ Prueba con otros hiperparámetros


In [ ]:
N_CUBETAS = 2**14       # Menos cubetas = más colisiones potenciales
T_TUPLA = 2
N_TABLAS = 20           # Más tablas = más oportunidades de colisión

tablas_ng_bin = [MinHashTable(N_CUBETAS, T_TUPLA, VOCMAX) for _ in range(N_TABLAS)]
sims, orden = busca_pares_documentos(bolsas_base_csr,
                                     bolsas_consultas_csr,
                                     bolsas_base,
                                     bolsas_consultas,
                                     tablas_ng_bin,
                                     similitud_jaccard)
print("------ C O N S U L T A ------\n", consultas[21])
print("\n------ M Á S  S I M I L A R ------\n", base[list(orden[21])[0]])


------ C O N S U L T A ------
 actually the book be call seventh day adventist believe and there be 27 basica belief i believe it be print by the reveiew and herald publishing association competition be the law of the jungle cooperation be the law of civilization eldridge cleaver


IndexError: list index out of range


Lanzo error. Esto significa que **no se encontró ningún documento similar** para esta consulta. Es decir, la lista orden[21] está vacía.

La posible causa es que no hubo colisión en las tablas, es decir, ningún documento cayó en la misma cubeta que la consulta, además, en general con solo 2 funciones hash, las firmas pueden no ser precisas


In [ ]:
N_CUBETAS = 2**16       # 65,536 cubetas
T_TUPLA = 2
N_TABLAS = 30           # Muchas tablas

tablas_ng_2 = [MinHashTable(N_CUBETAS, T_TUPLA, VOCMAX) for _ in range(N_TABLAS)]
sims3, orden2 = busca_pares_documentos(bolsas_base_csr,
                                       bolsas_consultas_csr,
                                       bolsas_base,
                                       bolsas_consultas,
                                       tablas_ng_2,
                                       similitud_jaccard)


print("------ C O N S U L T A ------\n", consultas[21])
print("\n------ M Á S  S I M I L A R ------\n", base[list(orden2[21])[0]])


------ C O N S U L T A ------
 actually the book be call seventh day adventist believe and there be 27 basica belief i believe it be print by the reveiew and herald publishing association competition be the law of the jungle cooperation be the law of civilization eldridge cleaver

------ M Á S  S I M I L A R ------
 do the word chill effect stimulate impulse within that small collection of neuron you call a brain cpk it be 80 day do you know where your wallet be


Aunque la consulta y el documento más similar parecen **temáticamente diferentes**, esto puede suceder porque ambas frases usan estructuras parecidas o tienen terminos que son muy frecuentes en común. Hay que tomar en cuenta que LSH encuentra coincidencias sintácticas, pero no entiende el significado profundo de las palabras, su significado real semántico.

Quizas experimentando con valores más altos en T_TUPLA (como 4 o 6) para ver si mejora la coherencia de los resultados.

In [ ]:
N_CUBETAS = 2**14       # 16,384 cubetas
T_TUPLA = 2
N_TABLAS = 25           # Bastantes tablas para mejorar el recall

tablas_ng_alt = [MinHashTable(N_CUBETAS, T_TUPLA, VOCMAX) for _ in range(N_TABLAS)]
sims_alt, orden3 = busca_pares_documentos(bolsas_base_csr,
                                             bolsas_consultas_csr,
                                             bolsas_base,
                                             bolsas_consultas,
                                             tablas_ng_alt,
                                             similitud_jaccard)

print("------ C O N S U L T A ------\n", consultas[21])
print("\n------ M Á S  S I M I L A R ------\n", base[orden3[21][0]])


------ C O N S U L T A ------
 actually the book be call seventh day adventist believe and there be 27 basica belief i believe it be print by the reveiew and herald publishing association competition be the law of the jungle cooperation be the law of civilization eldridge cleaver

⚠️ No se encontró ningún documento similar con esta configuración.


Aunque hay 25 tablas, la consulta puede no haber coincidido en ninguna cubeta útil y aunque haya colisionado, la similitud real pudo ser tan baja que fue descartada. Incrementar las N_TABLAS (Aunque dos documentos no caigan juntos en una tabla, pueden hacerlo en otra).

In [ ]:
N_CUBETAS = 2**8        # 256 cubetas indica alta colisión (poca discriminación)
T_TUPLA = 2
N_TABLAS = 5            # Muy pocas tablas indica bajo recall

tablas_ng_4 = [MinHashTable(N_CUBETAS, T_TUPLA, VOCMAX) for _ in range(N_TABLAS)]

sims4, orden4 = busca_pares_documentos(bolsas_base_csr,
                                       bolsas_consultas_csr,
                                       bolsas_base,
                                       bolsas_consultas,
                                       tablas_ng_4,
                                       similitud_jaccard)

print("------ C O N S U L T A ------\n", consultas[21])
print("\n------ M Á S  S I M I L A R ------\n", base[list(orden4[21])[0]])

Streaming output truncated to the last 5000 lines.
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Error, tabla llena!
¡Err

IndexError: list index out of range


**No se encontró ningún documento similar**.  
La lista orden4[21] está vacía.

Esto se debe a que las cubetas fueron insuficientes (N_CUBETAS=256), es decir, demasiados documentos colisionaron en las mismas cubetas.